# Половина 2: кросс-энкодер `bge-reranker-v2-m3`

Второй движок итогового бленда. База предобучена **как реранкер** — на парах
«два текста → логит релевантности», то есть ровно на нужной механике; по замерам
четырёх баз при одинаковом рецепте этот класс предобучения решает больше, чем
размер модели.

    база     BAAI/bge-reranker-v2-m3, 568M
    вход     576 токенов, атрибуты до 1024 символов, значение до 40
    доска    0.5382 в одиночку
    стадия B `_all` — обучение на ВСЕХ ручных фолдах, включая fold 0

Стадия B здесь идёт по замороженному рецепту и **без локальной валидации**:
fold 0 отдан в обучение, поэтому сравнивать вариации нечем, решает доска. Это
осознанный размен — половина 1 (`01_mmbert.ipynb`) валидируется локально и
страхует выбор.

Код половины 2 — `src/solution` и `src/utils`, половины 1 — `src/ecup`
и `src/scripts`; соединяются они в `03_blend.ipynb`.

## 1. Импорты

In [ ]:
import os
import pathlib
import sys

# Тетради лежат в корне репозитория — там же, где src/.
ROOT = pathlib.Path.cwd()
os.chdir(ROOT)
os.environ["ECUP_ROOT"] = str(ROOT)
sys.path.insert(0, str(ROOT))
print("корень:", ROOT)

from src import config
from src.solution.batch import merge, slice_folds
from src.solution.cross_encoder import CrossEncoder
from src.utils import bootstrap, data, llm, noise, packaging, runlog

## 2. Окружение и данные

In [ ]:
LOG_FILE = runlog.start('solution')
bootstrap.setup(group='all')
llm.ensure(n_train=10_300_000);

## 3. Конфигурация

In [ ]:
N_TRAIN = 10_300_000                     # 2М — разведка, 10.3М — весь train корпуса
MAX_LEN, ATTR_CHARS = 576, 1024
BATCH, ACCUM, BUCKET = 16, 8, 64         # шаг тот же: 16 × 8 = 128 пар

NAME = f'bge_{N_TRAIN // 1_000_000}m_len{MAX_LEN}'
MODEL_CLASS = f'{CrossEncoder.__module__}.{CrossEncoder.__name__}'

# модель — вход и раскладка батча, общие для обеих стадий
MODEL_KW = dict(
    base_model='BAAI/bge-reranker-v2-m3',
    max_len=MAX_LEN, attr_chars=ATTR_CHARS, val_chars=40,
    batch_size=BATCH, grad_accum=ACCUM, bucket=BUCKET,
    flip=False, use_category=True, rank_attrs=True,
)

# стадия A — LLM-пары
A_NAME = f'{NAME}_A'
STAGE_A = dict(lr=1e-5, epochs=1, schedule='linear')

# стадия B — ФИНАЛЬНЫЙ рецепт: оптимум зажат замерами со всех сторон
# (расширение гейта −0.0018, третья эпоха −0.0118, контекст дважды −0.0162)
B_NAME = f'{NAME}_B_all'
STAGE_B = dict(lr=1e-5, epochs=2, schedule='cosine')
REPLAY = 0.2
NOISE_KW = dict(
    w=0.1,                               # вес противоречивой метки в лоссе
    hi=0.8, lo=0.3,                      # жаккар имён: негатив шумный >= hi, позитив <= lo
    agree=0.8,                           # согласие атрибутов, снимающее противоречие
    gate=('Красота и гигиена',),         # категории, чья разметка глушится целиком
)

print(f'{NAME}: {MODEL_KW["base_model"]} · {N_TRAIN:,} пар · '
      f'вход {MAX_LEN} токенов / {ATTR_CHARS} символов')

## 4. Стадия A — машинная разметка

In [ ]:
model = CrossEncoder(**MODEL_KW, **STAGE_A)
model.fit(llm.stream('train', n=N_TRAIN, seed=7, chunk=1_000_000),
          checkpoint=config.ARTIFACTS / A_NAME)
model.save(config.ARTIFACTS / A_NAME)
del model

### Одиночный архив стадии A

In [ ]:
zip_a = packaging.build(approach=A_NAME, model_class=MODEL_CLASS)
packaging.verify(zip_a)

## 5. Стадия B — ручная разметка

In [ ]:
items, pairs = data.load_items(), data.load_folds()
human = slice_folds(items, pairs, pairs['fold'].to_numpy() >= 0)   # _all: fold 0 в обучении
human.weight = noise.weights(human, **NOISE_KW)

model = CrossEncoder.load(config.ARTIFACTS / A_NAME).retune(**STAGE_B)
model.fit(merge(human, llm.load('train', n=int(REPLAY * human.n))),
          checkpoint=config.ARTIFACTS / B_NAME)
model.save(config.ARTIFACTS / B_NAME)
del model

### Одиночный архив стадии B

Это и есть вторая половина бленда. Дальше — `03_blend.ipynb`.

In [ ]:
zip_b = packaging.build(approach=B_NAME, model_class=MODEL_CLASS)
packaging.verify(zip_b)
print('половина 2 готова:', config.ARTIFACTS / B_NAME)